# Análise de Despesas

Este notebook demonstra como criar agentes que utilizam plugins para processar despesas de viagem a partir de imagens locais de recibos, gerar um email de pedido de reembolso e visualizar os dados de despesas utilizando um gráfico de pizza. Os agentes escolhem dinamicamente funções com base no contexto da tarefa.

Passos:
1. O Agente OCR processa a imagem local do recibo e extrai os dados das despesas de viagem.
2. O Agente de Email gera um email para o pedido de reembolso.

### Exemplo de um cenário de despesas de viagem:
Imagine que é um colaborador a viajar para uma reunião de negócios noutra cidade. A sua empresa tem uma política para reembolsar todas as despesas de viagem razoáveis. Aqui está uma discriminação das despesas de viagem potenciais:
- Transporte:
Passagem aérea para uma viagem de ida e volta da sua cidade de origem à cidade de destino.
Táxi ou serviços de ride-hailing para ir e voltar do aeroporto.
Transporte local na cidade de destino (como transporte público, carros de aluguer ou táxis).

- Alojamento:
Estadia em hotel durante três noites num hotel de negócios de gama média próximo do local da reunião.

- Refeições:
Subsídio diário para pequeno-almoço, almoço e jantar, baseado na política de diárias da empresa.

- Despesas Diversas:
Taxas de estacionamento no aeroporto.
Custos de acesso à internet no hotel.
Gorjetas ou pequenas taxas de serviço.

- Documentação:
Submete todos os recibos (voos, táxis, hotel, refeições, etc.) e um relatório de despesas preenchido para reembolso.


## Importar bibliotecas necessárias

Importe as bibliotecas e módulos necessários para o notebook.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## Definir Modelos de Despesa

 Crie um modelo Pydantic para despesas individuais e uma classe ExpenseFormatter para converter uma consulta do utilizador em dados de despesa estruturados.

 Cada despesa será representada no formato:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Defining Tools - Gerar o Email

Crie uma função de ferramenta para gerar um email para submeter uma reclamação de despesas.
- Esta ferramenta utiliza o decorador `@tool` do Microsoft Agent Framework.
- Calcula o valor total das despesas e formata os detalhes num corpo de email.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Ferramenta para Extrair Despesas de Viagem de Imagens de Recibos

Crie uma função de ferramenta para extrair despesas de viagem de imagens de recibos.  
- Esta ferramenta utiliza o decorador `@tool` do Microsoft Agent Framework.  
- Lê a imagem do recibo, codifica-a como base64 e retorna o URI de dados para o agente analisar.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Processamento de Despesas

Defina os agentes e ligue-os numa sequência de fluxo de trabalho usando `WorkflowBuilder`.
- O agente OCR extrai dados estruturados de despesas a partir da imagem do recibo usando a ferramenta `load_receipt_image`.
- O agente Email pega nos dados extraídos e gera um email profissional de reclamação de despesas usando a ferramenta `generate_expense_email`.
- O `WorkflowBuilder` com `add_edge` cria um pipeline sequencial: Agente OCR → Agente Email.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Função principal

Construa o fluxo de trabalho sequencial e execute-o para processar a imagem do recibo e gerar o e-mail de reembolso de despesas.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos empenhemos em garantir a precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original no seu idioma nativo deve ser considerado a fonte oficial. Para informações críticas, recomenda-se a tradução profissional realizada por humanos. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações erradas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
